# Agentic Email Topic Classification Demo

This notebook demonstrates a topic classification pipeline for corporate emails using OpenAI's GPT models and the Enron public email dataset.

## Prerequisites
----------------

Before running the notebook, ensure you have:

1. **OpenAI API Access**
   - A valid OpenAI API key from an account with billing enabled.

2. **Kaggle Account**
   - An account with API access enabled to download the dataset.

## Step 1: Set Required Environment Variables
--------------------------------------------

Create a `.env` file in the project root directory with the following content:

    # OpenAI configuration
    MODEL_OPENAI_MODEL_NAME=gpt-4-turbo
    MODEL_OPENAI_OPENAI_API_KEY=your_openai_api_key_here

    # Kaggle configuration
    KAGGLE_USERNAME=your_kaggle_username_here
    KAGGLE_KEY=your_kaggle_key_here

💡 Replace the placeholders with your actual OpenAI and Kaggle credentials.

How to get your Kaggle API credentials:
1. Go to https://www.kaggle.com/account
2. Scroll down to the API section
3. Click "Create New API Token"
4. Open the downloaded `kaggle.json` file and copy the `username` and `key` into the `.env`

## Step 2: What the Notebook Does
---------------------------------

The notebook performs the following steps:

- Downloads the Enron Email Dataset from Kaggle
- Preprocesses raw email text into a clean input format
- Initializes an agentic classifier using OpenAI's GPT models
- Automatically generates and updates topics by analyzing semantic content of emails
- Classifies new emails into the most relevant topic or creates a new one if necessary

### Cost Estimate
----------------

Running the notebook with `gpt-4-turbo` incurs an estimated cost of ~$2 USD, depending on the number of emails processed.

💡 You can reduce cost by:
- Using `gpt-3.5-turbo`
- Limiting the number of processed emails by adjusting the `N_DOCUMENTS` parameter

## Dataset Source
-----------------

- Enron Email Dataset  
  Provided by Kaggle: https://www.kaggle.com/datasets/wcukierski/enron-email-dataset

## Notes
--------

- The classifier uses a fully unsupervised, agent-based approach with no predefined topics.
- Topic generation and refinement is done on the fly based on email contents.


In [ ]:
import os
import email

from dotenv import load_dotenv

load_dotenv()

# from google.colab import userdata
KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv('KAGGLE_KEY')

import kaggle as kg
import pandas as pd

from agentic_text_cls import AgenticTextCls
from settings import ModelSettings, AgentSettings
from logging_config import setup_logging, logger

In [ ]:
DATASET_NAME = "wcukierski/enron-email-dataset"
N_DOCUMENTS = 200

In [ ]:
kg.api.dataset_download_files(dataset=DATASET_NAME, path='data', unzip=True)
df_emails = pd.read_csv('./data/emails.csv')
df_emails.info()

In [ ]:
# Helper functions
def get_text_from_email(msg):
    '''To get the content from email objects'''
    parts = []
    for part in msg.walk():
        if part.get_content_type() == 'text/plain':
            parts.append( part.get_payload() )
    return ''.join(parts)

def split_email_addresses(line):
    '''To separate multiple email addresses'''
    if line:
        addrs = line.split(',')
        addrs = frozenset(map(lambda x: x.strip(), addrs))
    else:
        addrs = None
    return addrs

# Parse the emails into a list email objects
messages = list(map(email.message_from_string, df_emails['message']))
df_emails.drop('message', axis=1, inplace=True)
# Get fields from parsed email objects
keys = messages[0].keys()
for key in keys:
    df_emails[key] = [doc[key] for doc in messages]
# Parse content from emails
df_emails['content'] = list(map(get_text_from_email, messages))
# Split multiple email addresses
df_emails['From'] = df_emails['From'].map(split_email_addresses)
df_emails['To'] = df_emails['To'].map(split_email_addresses)

# Extract the root of 'file' as 'user'
df_emails['user'] = df_emails['file'].map(lambda x:x.split('/')[0])
del messages

df_emails.head()

In [ ]:
# Helper function to prepare input for classification
def preprare_input_text(from_field: str, to_field, subject: str, content: str) -> str:
    return f"""From: {from_field}\nTo: {to_field}\nSubject: {subject}\n
    {content}
    """

sample_msg = df_emails[["From", "To", "Subject", "content"]].sample(N_DOCUMENTS, random_state=42).values.tolist()
input_test_list = []
for msg in sample_msg:
    from_field, to_field, subject, content = msg[0], msg[1], msg[2], msg[3]
    from_field = "".join([i for i in from_field])
    to_field = ";".join([i for i in to_field]) if to_field is not None else ""
    input_text = preprare_input_text(from_field=from_field, to_field=to_field, subject=subject, content=content)
    input_test_list.append(input_text)

## Run Agent

In [ ]:
agentic_config = AgentSettings()
setup_logging(agentic_config)
configs = ModelSettings()
ac = AgenticTextCls(
    model_settings=configs, logger=logger, generate_new_metadata_idx=True
)

In [ ]:
for idx_txt, text in enumerate(input_test_list):
    print(f"Processing: {idx_txt} ...")
    ac.classify_text(text=text)

logger.info(f"Dict with topics:\n{ac.topics}")

In [ ]:
# Store results
import json

with open("./output.json", "w") as f:
    json.dump(ac.topics, f)